### 하이퍼파라미터
* 예측력을 높이기 위해 사용
* 과대/과소 적합을 방지하기 위해 사용
* 사용자가 지정하는 파라미터

#### 하이퍼파라미터 튜닝
* 특정 알고리즘의 매개변수 값을 변경하면서 최적의 파라미터를 찾는 방식
* GridSearchCV클래스를 통해 하이퍼파라미터 탐색과 교차 검증을 한번에 수행

#### GridSearchCV
* 교차 검증으로 하이퍼파라미터 탐색을 수행한다
* 최상의 모델을 찾은 후 훈련 세트 전체를 사용해 최종 모델을 훈련한다
* 매개변수
    - GridSearchCV(모델, param_grid = 파라미터, cv=반복횟수, n_jobs=코어 수(-1, 모든 코어 사용))

### K-최근접 이웃 알고리즘
* 주위에서 가장 가까운 다른 데이터를 보고 현재 데이터를 판단

#### 하이퍼파라미터
* n_neighbors
    - 기본 가까운 5개의 데이터를 보고 자기 자신이 어디에 속하는지를 판단
    - 비교하고자 하는 데이터의 수가 적을 수록 과대 적합이 된다
* metric : 거리계산 척도
    - euclidean(녹색) : 유클리디안 거리 측정
        * 목표 지점까지 가장 짧은 거리
        * 유클리디안은 일반적인 연속형 데이터에 적합
    - manhattan(빨간색) : 맨하튼 거리 측정 방법
        * 찾아가는 경로의 모든 길이의 합
        * 맨하탄 거리는 이상치 영향을 줄이고 싶을 때 사용
* weights : 가중치
    - uniform : 거리에 가중치 부여하지 않음(균일한 가중치)
    - distance : 거리에 가중치 부여(가까이 있는 데이터에 가중치)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data_set/5.스케일링/titanic_cleaning.csv")
df.head(2)

In [ ]:
df.columns

In [ ]:
f = ['PassengerId', 'Sex', 'Age', 'SibSp', 'Parch','Fare']
label = 'Survived'
X, y = df[f], df[label]

In [ ]:
minMaxScaler = MinMaxScaler()
minMaxScaler.fit( X )
X_scaler = minMaxScaler.transform( X )
X_scaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaler, y, test_size=0.2)

In [ ]:
knn = KNeighborsClassifier()
knn.fit( X_train, y_train )
print("train : ", knn.score( X_train, y_train))
print("test : ", knn.score( X_test, y_test))

In [ ]:
k_param = range(1 ,11)
k_param
train_list = []
test_list = []

for k in k_param:
    #print( k )
    knn = KNeighborsClassifier(n_neighbors = k)
    knn.fit(X_train, y_train)
    train_score = knn.score(X_train, y_train)
    test_score = knn.score(X_test, y_test)
    train_list.append(train_score)
    test_list.append(test_score)

In [ ]:
print(train_list)
print(test_list)

In [ ]:
dic = {
    "k" : k_param,
    "train정확도": train_list,
    "test정확도": test_list
}
score_df = pd.DataFrame( dic )
score_df

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic' #그래프 내 한글 표현.

fig, ax = plt.subplots(figsize=(5,5))
sns.lineplot(x='k', y='train정확도', data=score_df)
sns.lineplot(x='k', y='test정확도', data=score_df)
plt.show()

In [ ]:
params = {
    'n_neighbors':k_param
}
knn = KNeighborsClassifier()
grid_cv = GridSearchCV(knn, param_grid = params, cv = 5, n_jobs = -1)

In [ ]:
grid_cv.fit(X_train, y_train)

In [ ]:
grid_cv.best_params_

In [ ]:
print("train score : ", grid_cv.score(X_train, y_train))
print("test score : ", grid_cv.score(X_test, y_test))

In [ ]:
params = {
    'n_neighbors':k_param,
    'metric': ['manhattan', 'euclidean'],
    'weights':['uniform','distance']
}
knn = KNeighborsClassifier()
grid_cv = GridSearchCV(knn, param_grid = params, cv = 5, n_jobs = -1)

In [ ]:
grid_cv.fit(X_train, y_train)
print("train score : ", grid_cv.score(X_train, y_train))
print("test score : ", grid_cv.score(X_test, y_test))
grid_cv.best_params_

In [ ]:
knn = KNeighborsClassifier(metric='euclidean', n_neighbors=4, weights='uniform')

In [ ]:
knn.fit(X_train,y_train)
print("train score : ", knn.score(X_train, y_train))
print("test score : ", knn.score(X_test, y_test))

### svm 하이퍼 파라미터
* C(cost)
    - cost값이 작으면 과소적합이 발생한다.
        * cost값을 작게하면 훈련 데이터에 대해 어느정도 오류를 허용하게 되며, 새로 들어오는 데이터는 잘 분류할 수 있다
    - cost값이 커지면 과대적합이 발생한다.
        * cost값을 크게하면 훈련 데이터에 대해 오류가 최소화 되지만, 새로 들어오는 데이터는 분류를 잘못할 가능성이 높다는 것이다
* gamma
    - 결정 경계에 영향을 끼치는 범위를 조절하는 변수
    - gamma가 크면 과대적합이 될 수 있다
        - gamma가 크면 결정경계에 영향력이 커 결정경계가 복잡해진다
    - gamma가 작으면 과소적합이 될 수 있다.
        - gamma가 작으면 결정경계에 영향력이 미미하여 결정경계가 부드러워진다

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv("../data_set/5.스케일링/titanic_cleaning.csv")
df.head(2)

In [ ]:
df.columns

In [ ]:
f = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch','Fare']
label = 'Survived'
X,y = df[f], df[label]

In [ ]:
scaler = StandardScaler()
scaler.fit(X)
X_scaler = scaler.transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaler, y, test_size=0.2)

In [ ]:
svc = SVC(kernel = "linear")
svc.fit(X_train, y_train)

In [ ]:
print("train score : ", svc.score(X_train, y_train))
print("train score : ", svc.score(X_test, y_test))

In [ ]:
svc = SVC(kernel = "linear", C = 100)
svc.fit(X_train, y_train)

print("train score : ", svc.score(X_train, y_train))
print("train score : ", svc.score(X_test, y_test))

In [ ]:
param_cost = [0.01, 0.1, 1, 10, 100]
train_list = []
test_list = []

for cost in param_cost:
    svc = SVC(kernel="linear", C = cost)
    svc.fit(X_train, y_train)
    train_list.append(svc.score(X_train, y_train))
    test_list.append(svc.score(X_test, y_test))

In [ ]:
dic = {
    "param": param_cost,
    "train정확도":train_list,
    "test정확도":test_list
}
score_df = pd.DataFrame(dic)
score_df

In [ ]:
param_cost = [0.01, 0.1, 1, 10, 100]
param_gamma = [0.001, 0.01, 0.1, 1, 10]
train_list = []
test_list = []

for cost in enumerate(param_cost):
    svc = SVC(kernel="rbf", C = cost, gamma = param_gamma[idx])
    svc.fit(X_train, y_train)
    train_list.append(svc.score(X_train, y_train))
    test_list.append(svc.score(X_test, y_test))

dic = {
    "param": param_cost,
    'param_gamma' : param_gamma,
    "train정확도":train_list,
    "test정확도":test_list
}
score_df = pd.DataFrame(dic)
score_df

In [ ]:
param_cost = [0.01, 0.1, 1, 10, 100]
param_gamma = [0.001, 0.01, 0.1, 1, 10]

In [ ]:
params = {
    "C":param_cost,
    "gamma":param_gamma
}

In [ ]:
svc = SVC(kernel="rbf")

In [ ]:
grid_cv = GridSearchCV(svc, param_grid=params, cv=3, n_jobs = -1)

In [ ]:
grid_cv.fit(X_train, y_train)
grid_cv.best_params_

In [ ]:
print("train score : ", grid_cv.score(X_train, y_train))
print("test score : ", grid_cv.score(X_test, y_test))

### 랜덤 포레스트 하이퍼 파라미터
* n_estimators
    - 트리의 개수. default=10
    - 트리의 개수가 많이면 성능은 좋아지지만, 시간이 오래 걸릴수 있다.
* max_depth
    - 트리의 최대 깊이. default = None
    - 완벽하게 파라미터값이 결정될 때 까지 분할 또는 min_samples_split보다 작아질 때까지 분할
    - 깊이가 깊어지면 과대적합될 수 있으므로 적절히 제어 필요
* min_samples_leaf
    - 맆노드(자식노드가 없는 노드)가 되기 위해 최소한의 샘플 데이터 수
    - min_samples_split과 함께 과대적합 제어용도
    - 불균형 데이터가 있는 경우 작게 설정하는게 좋다
* min_samples_split
    - 노드를 분할하기 위한 최소한의 샘플 데이터수. default = 2
    - 과적합을 제어하는데 사용
    - 작게 설정할 수록 분할 노드가 많아져 과대적합 가능성 증가

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

In [ ]:
df.head(2)
#타이타닉 데이터 셋

In [ ]:
#X,y 타이타닉 데이터
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
clf = DecisionTreeClassifier(max_depth = 2) #max_depth = 2
clf = clf.fit(X_train, y_train)
fig = plt.figure(figsize=(15,10), facecolor="white")
tree.plot_tree(clf, feature_names=f, class_names=list(label))
plt.show()

In [ ]:
rfc = RandomForestClassifier(
    n_estimators=100, max_depth=10, min_samples_leaf=4, min_samples_split=100
)

rfc_def = RandomForestClassifier()

In [ ]:
rfc.fit(X_train, y_train)
print("train : ", rfc.score(X_train, y_train))
print("test : ", rfc.score(X_test, y_test))

In [ ]:
rfc_def.fit(X_train, y_train)
print("train : ", rfc_def.score(X_train, y_train))
print("test : ", rfc_def.score(X_test, y_test))

In [ ]:
params = {
    "n_estimators" : range(10,101,10),
    "max_depth" : range(4,11,2),
    "min_samples_leaf" : range(5,21,5),
    "min_samples_split" : range(4,21,4)
}
rfc = RandomForestClassifier()

In [ ]:
grid_cv = GridSearchCV(rfc, param_grid = params, cv = 3, n_jobs=-1)
grid_cv.fit(X_train,y_train)

In [ ]:
print("최적의 파라미터 : ", grid_cv.best_params_)
print("train : ", grid_cv.score(X_train, y_train))
print("test : ", grid_cv.score(X_test, y_test))

In [ ]:
df.head(2)

In [ ]:
inputData = [[3,0,22.0,1,0,7.2500]]

In [ ]:
grid_cv.predict(inputData)

### 하이퍼파라미터
* n_estimators
    - 학습기의 수 설정.
    - 많으면 예측률은 올라가나 시간이 오래걸릴 수 있음.
* learning_rate
    - GBM이 학습을 진행할 때마다 적용하는 학습률로서 순차적으로 오류 값을 보정해가는 데 적용하는 수 이다
    - 0~1 사이의 값으로 지정한다. default=0.1
    - 너무 작은 값이면 업데이트를 너무 많이 하게 되어 꼼꼼하지만 시간이 오래 걸린다
    - 너무 큰 값이면 최소 오류 값을 찾지 못할 수 있지만 빠르다
* subsample
    - 학습기가 학습에 사용하는 데이터 샘플의 비율(0~1). default=1.
    - 0.5일경우 50%학습데이터를 사용한다는 것이다
    - 과대적합을 줄이려면 작은 값을 적용해야 한다.

In [1]:
from sklearn.ensemble import GradientBoostingClassifier

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

NameError: name 'train_test_split' is not defined

In [ ]:
gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)
print("train : ", gbc.score(X_train, y_train))
print("test : ", gbc.score(X_test, y_test))

In [ ]:
import numpy as np

In [ ]:
params = {
    'learning_rate' : [0.1, 0.3, 0.5, 0.7, 0.9],
    'n_estimators': [100,300,500],
    'subsample' : np.arange(0.1, 1, 0.2) #실수값은 np활용
}

In [ ]:
grid_cv = GridSearchCV(gbc, param_grid = params, cv=3, n_jobs = -1)

In [ ]:
grid_cv.fit(X_train, y_train)

In [ ]:
print("최적의 파라미터 : ", grid_cv.best_params_)
print("train : ", grid_cv.score(X_train, y_train))
print("test : ", grid_cv.score(X_test, y_test))

In [ ]:
print(gbc.feature_importances_)
X_train.head(1)